# 02 — Serve (vLLM on T4) + completions + benchmark (+ demo tunnel)
Thin runner. Serves the merged model with vLLM, generates the base/tuned completion artifacts over the full held-out test set, benchmarks throughput, and (optionally) opens a cloudflared tunnel for demo-video recording.

T4 flags: `--dtype float16` (no bf16 on sm75), xformers attention fallback.

In [ ]:
%pip install -q -U vllm openai pandas
!git clone https://github.com/ahmedraza-96/ticket-triage-llm.git /kaggle/working/repo || git clone https://github.com/ahmedraza-96/ticket-triage-llm.git /content/repo
import os
REPO = '/kaggle/working/repo' if os.path.exists('/kaggle/working/repo') else '/content/repo'
os.chdir(REPO)
!python data/prepare_data.py && python data/make_eval_subsets.py
!pip freeze | grep -Ei '^vllm|^torch' | head

In [ ]:
MODEL = 'ahmedraza-96/Qwen3-4B-Instruct-2507-ticket-triage'   # tuned arm; rerun with base for arm 2
ARM = 'tuned'                                                  # or 'base' with Qwen/Qwen3-4B-Instruct-2507
import subprocess, time, urllib.request
server = subprocess.Popen(['vllm', 'serve', MODEL, '--dtype', 'float16',
                           '--max-model-len', '4096', '--gpu-memory-utilization', '0.85',
                           '--host', '0.0.0.0', '--port', '8000'],
                          stdout=open('vllm.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(120):
    try:
        urllib.request.urlopen('http://localhost:8000/v1/models'); print('vLLM up'); break
    except Exception: time.sleep(5)
else: raise RuntimeError('vLLM failed to start — check vllm.log')

In [ ]:
# Completions over the full held-out test set (~20-35 min) + throughput benchmark
!python infer/generate_completions.py --arm $ARM --model $MODEL
!python infer/benchmark_throughput.py --arm $ARM --model $MODEL
# Stash artifacts as kernel outputs for download
!cp -r artifacts /kaggle/working/artifacts_out 2>/dev/null || true

In [ ]:
# OPTIONAL (demo recording only): expose the endpoint via cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
import subprocess, re, time
tun = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in tun.stdout:
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if m: print('TUNNEL URL:', m.group(0)); break